<a href="https://colab.research.google.com/github/mxls34/AdvanceDatabase/blob/main/Chapter6_Indexing_and_Query_Optimization.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Chapter 6: Indexing & Query Optimization

ใน Lab นี้เราจะใช้ **PostgreSQL บน Neon** เพื่อดูผลของ Index ต่อความเร็วของ Query

*Neon คือ ผู้ให้บริการ Serverless PostgreSQL บนคลาวด์*

**สิ่งที่ต้องเตรียมก่อนเริ่ม**
1. สมัคร Neon (neon.tech) และสร้าง Project ใหม่
2. คัดลอก Connection String มาเก็บใน Colab Secrets ชื่อ `NEON_CONNECTION_STRING`
3. เปิดใช้งาน (toggle) ให้ notebook นี้เข้าถึง secret ได้

**Domain ที่ใช้ใน Lab**

ข้อมูลการเดินทาง (Trip) ของ App เรียกรถ — ตาราง `trips` ขนาด 500,000 แถว เพื่อให้เห็นผลของ Index ชัดเจน


## Setup & เชื่อมต่อฐานข้อมูล

In [4]:
!pip install psycopg2-binary pandas -q

In [7]:
import psycopg2
import pandas as pd
import os
import time

try:
    from google.colab import userdata
    CONN_STRING = userdata.get('NEON_CONNECTION_STRING')
except Exception:
    CONN_STRING = os.environ.get('NEON_CONNECTION_STRING')

conn = psycopg2.connect(CONN_STRING)
conn.autocommit = True
print("เชื่อมต่อสำเร็จ!")

เชื่อมต่อสำเร็จ!


### Helper Functions

- `run(sql)` — รัน SQL แล้วคืนผลลัพธ์เป็น pandas DataFrame
- `explain(sql, label)` — รัน `EXPLAIN ANALYZE` แล้วพิมพ์ผลลัพธ์แบบอ่านง่าย

In [8]:
import psycopg2

def get_connection():
    """สร้าง connection ใหม่"""
    c = psycopg2.connect(CONN_STRING)
    c.autocommit = True
    return c

conn = get_connection()

def run(sql, params=None, _retry=True):
    """รัน SQL แล้วคืนผลลัพธ์เป็น DataFrame — reconnect อัตโนมัติถ้า connection หลุด"""
    global conn
    try:
        with conn.cursor() as cur:
            cur.execute(sql, params)
            if cur.description:
                colnames = [d[0] for d in cur.description]
                return pd.DataFrame(cur.fetchall(), columns=colnames)
            return None
    except (psycopg2.OperationalError, psycopg2.InterfaceError) as e:
        if _retry:
            print(f"Connection หลุด ({e}) — กำลังเชื่อมต่อใหม่...")
            conn = get_connection()
            return run(sql, params, _retry=False)  # ลองใหม่แค่ 1 ครั้ง กันวนลูปไม่รู้จบ
        else:
            raise  # ถ้า reconnect แล้วยัง error อีก ให้ throw error จริงออกไป

def explain(sql, params=None, label=""):
    df = run("EXPLAIN ANALYZE " + sql, params)
    if label:
        print(f"=== {label} ===")
    for line in df.iloc[:, 0]:
        print(line)
    print()

## Schema: ตาราง `drivers`, `riders`, `trips`

**Domain:** ข้อมูลการเดินทาง (Trip) ของแอปเรียกรถ

### ตาราง `drivers`

| คอลัมน์ | ชนิดข้อมูล | คำอธิบาย |
|---|---|---|
| `driver_id` | `INT PRIMARY KEY` | รหัสคนขับ |
| `driver_name` | `TEXT NOT NULL` | ชื่อคนขับ |
| `city` | `TEXT NOT NULL` | เมืองที่คนขับประจำอยู่ |

### ตาราง `riders`

| คอลัมน์ | ชนิดข้อมูล | คำอธิบาย |
|---|---|---|
| `rider_id` | `INT PRIMARY KEY` | รหัสผู้โดยสาร |
| `rider_name` | `TEXT NOT NULL` | ชื่อผู้โดยสาร |

### ตาราง `trips` (ตารางหลัก — 500,000 แถว)

| คอลัมน์ | ชนิดข้อมูล | คำอธิบาย |
|---|---|---|
| `trip_id` | `BIGSERIAL PRIMARY KEY` | รหัสการเดินทาง |
| `driver_id` | `INT NOT NULL` | รหัสคนขับ (ใช้ทำ index หลัก) |
| `rider_id` | `INT NOT NULL` | รหัสผู้โดยสาร |
| `pickup_zone` | `TEXT NOT NULL` | โซนรับผู้โดยสาร (low-cardinality ~10 โซน) |
| `trip_status` | `TEXT NOT NULL` | สถานะ: 'completed', 'cancelled', 'in_progress' |
| `trip_date` | `TIMESTAMP NOT NULL` | วันเวลาที่เดินทาง (ใช้สาธิต sargability) |
| `fare_amount` | `NUMERIC(8,2) NOT NULL` | ค่าโดยสาร |
| `distance_km` | `NUMERIC(6,2) NOT NULL` | ระยะทาง (กม.) |

### SQL DDL

```sql
CREATE TABLE drivers (
    driver_id   INT PRIMARY KEY,
    driver_name TEXT NOT NULL,
    city        TEXT NOT NULL
);

CREATE TABLE riders (
    rider_id   INT PRIMARY KEY,
    rider_name TEXT NOT NULL
);

CREATE TABLE trips (
    trip_id      BIGSERIAL PRIMARY KEY,
    driver_id    INT NOT NULL,
    rider_id     INT NOT NULL,
    pickup_zone  TEXT NOT NULL,
    trip_status  TEXT NOT NULL,
    trip_date    TIMESTAMP NOT NULL,
    fare_amount  NUMERIC(8,2) NOT NULL,
    distance_km  NUMERIC(6,2) NOT NULL
);
```

### สร้าง Schema

In [9]:
run("""
DROP TABLE IF EXISTS trips;
DROP TABLE IF EXISTS drivers;
DROP TABLE IF EXISTS riders;

CREATE TABLE drivers (
    driver_id   INT PRIMARY KEY,
    driver_name TEXT NOT NULL,
    city        TEXT NOT NULL
);

CREATE TABLE riders (
    rider_id   INT PRIMARY KEY,
    rider_name TEXT NOT NULL
);

CREATE TABLE trips (
    trip_id      BIGSERIAL PRIMARY KEY,
    driver_id    INT NOT NULL,
    rider_id     INT NOT NULL,
    pickup_zone  TEXT NOT NULL,
    trip_status  TEXT NOT NULL,
    trip_date    TIMESTAMP NOT NULL,
    fare_amount  NUMERIC(8,2) NOT NULL,
    distance_km  NUMERIC(6,2) NOT NULL
);
""")
print("สร้างตารางสำเร็จ")

สร้างตารางสำเร็จ


### Seed ข้อมูล

ขั้นตอนนี้ seed ข้อมูล 500,000 แถวลงตาราง `trips` ผ่านเครือข่ายไปยัง Neon

จะใช้ `COPY` แทน `INSERT` ทีละแถว

In [10]:
import random
import io
from datetime import datetime, timedelta

random.seed(42)

#ข้อมูลสำหรับการสุ่ม
CITIES = ["Bangkok", "Chiang Mai", "Phuket", "Khon Kaen", "Prachinburi"] #5 cities
ZONES = ["Zone-A", "Zone-B", "Zone-C", "Zone-D", "Zone-E", "Zone-F", "Zone-G", "Zone-H", "Zone-I", "Zone-J"] #10 zones
STATUSES = ["completed", "completed", "completed", "completed", "cancelled", "in_progress"]  #เจตนาให้การสุ่มได้ completed เยอะสุด ประมาณ 66.7%

N_DRIVERS = 500
N_RIDERS = 5000
N_TRIPS = 500_000 #Python ใช้ _ แทน , คั่นหลักพัน

t0 = time.time()

with conn.cursor() as cur:
    driver_rows = [(i, f"Driver_{i}", random.choice(CITIES)) for i in range(1, N_DRIVERS + 1)]
    cur.executemany("INSERT INTO drivers (driver_id, driver_name, city) VALUES (%s, %s, %s)", driver_rows)

    rider_rows = [(i, f"Rider_{i}") for i in range(1, N_RIDERS + 1)]
    cur.executemany("INSERT INTO riders (rider_id, rider_name) VALUES (%s, %s)", rider_rows)

print(f"Seed drivers/riders เสร็จใน {time.time()-t0:.1f} วินาที")

Seed drivers/riders เสร็จใน 435.9 วินาที


In [11]:
def gen_trips_chunk(n, start_date=datetime(2025, 1, 1)):
    buf = io.StringIO()
    for _ in range(n):
        driver_id = random.randint(1, N_DRIVERS)
        rider_id = random.randint(1, N_RIDERS)
        zone = random.choice(ZONES)
        status = random.choice(STATUSES)
        days_offset = random.randint(0, 600)
        seconds_offset = random.randint(0, 86399)
        trip_date = start_date + timedelta(days=days_offset, seconds=seconds_offset)
        fare = round(random.uniform(30, 500), 2)
        distance = round(random.uniform(0.5, 45), 2)
        buf.write(f"{driver_id}\t{rider_id}\t{zone}\t{status}\t{trip_date}\t{fare}\t{distance}\n")
    buf.seek(0)
    return buf

t1 = time.time()
CHUNK = 50_000
total = 0
with conn.cursor() as cur:
    while total < N_TRIPS:
        n = min(CHUNK, N_TRIPS - total)
        buf = gen_trips_chunk(n)
        cur.copy_expert(
            "COPY trips (driver_id, rider_id, pickup_zone, trip_status, trip_date, fare_amount, distance_km) FROM STDIN",
            buf
        )
        total += n
        print(f"  seed แล้ว {total:,} / {N_TRIPS:,} trips...")

print(f"Seed trips เสร็จใน {time.time()-t1:.1f} วินาที")

  seed แล้ว 50,000 / 500,000 trips...
  seed แล้ว 100,000 / 500,000 trips...
  seed แล้ว 150,000 / 500,000 trips...
  seed แล้ว 200,000 / 500,000 trips...
  seed แล้ว 250,000 / 500,000 trips...
  seed แล้ว 300,000 / 500,000 trips...
  seed แล้ว 350,000 / 500,000 trips...
  seed แล้ว 400,000 / 500,000 trips...
  seed แล้ว 450,000 / 500,000 trips...
  seed แล้ว 500,000 / 500,000 trips...
Seed trips เสร็จใน 9.1 วินาที


In [12]:
run("SELECT count(*) AS total_trips FROM trips")

,total_trips
0,500000


**SET max_parallel_workers_per_gather = 0**

กำหนดให้ ยังไม่ใช้ความสามารถของ PostgreSQL ในเรื่องของ Parallel Query

ปกติถ้าตารางใหญ่พอ (แบบ trips ที่มี 500,000 แถว) PostgreSQL อาจเลือกแบ่งงานให้หลาย worker ช่วยกันสแกนพร้อมกัน (Parallel Seq Scan)

เป้าหมายของ Lab นี้คือให้เห็น Seq Scan ธรรมดา vs Index Scan ชัด ๆ ไม่อยากให้ concept "Parallel Query" มาแทรก

**ANALYZE**

สั่งให้ PostgreSQL เก็บสถิติ (statistics) ของแต่ละตาราง — จำนวนแถว, การกระจายของค่าในแต่ละคอลัมน์ ฯลฯ  

หลัง COPY ข้อมูลจำนวนมากเข้าไปใหม่ ๆ (เพิ่งทำเสร็จจากเซลล์ seed) PostgreSQL ยังไม่รู้ ว่าตอนนี้ตารางมีข้อมูลเยอะแค่ไหน เพราะ Autovacuum ยังไม่ทันมาอัปเดตให้ ถ้าไม่รัน ANALYZE เอง planner อาจยังคิดว่าตารางว่างเปล่าหรือมีข้อมูลน้อย แล้วเลือกแผนผิดเพี้ยนไป (เช่น เลือก Seq Scan ทั้งที่ควรใช้ Index Scan หรือกลับกัน) ทำให้ตัวเลขที่เห็นใน Lab ไม่ตรงกับที่ควรจะเป็น




In [ ]:
run("SET max_parallel_workers_per_gather = 0")
run("ANALYZE trips")
run("ANALYZE drivers")
run("ANALYZE riders")
print("ตั้งค่าเรียบร้อย — พร้อมเริ่ม Lab A")

ตั้งค่าเรียบร้อย — พร้อมเริ่ม Lab A



* **EXPLAIN** จะแสดงแผน ที่ planner ตัดสินใจว่าจะใช้ — เป็นการประมาณการ cost/rows ไม่ได้รัน query จริง

* **EXPLAIN ANALYZE** รัน query จริง แล้วรายงานเวลาจริง/จำนวนแถวจริง เทียบกับที่ประมาณไว้ จะใช้สำหรับเทียบผล before/after สร้าง index

---
## Lab A: Baseline — วัดผลก่อนมี Index

ก่อนสร้าง index ใด ๆ เลย มาดูกันว่า query หาข้อมูลของ driver คนหนึ่งจากตาราง 500,000 แถวช้าแค่ไหน



In [13]:
explain("SELECT * FROM trips WHERE driver_id = 250", label="Lab A: driver_id = 250 (ไม่มี index)")

=== Lab A: driver_id = 250 (ไม่มี index) ===
Gather  (cost=1000.00..9386.57 rows=994 width=53) (actual time=0.258..36.511 rows=998.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared hit=5683
  ->  Parallel Seq Scan on trips  (cost=0.00..8287.17 rows=414 width=53) (actual time=0.054..26.234 rows=332.67 loops=3)
        Filter: (driver_id = 250)
        Rows Removed by Filter: 166334
        Buffers: shared hit=5683
Planning:
  Buffers: shared hit=27 dirtied=1
Planning Time: 0.114 ms
Execution Time: 36.591 ms



**จากผลลัพธ์ที่ได้**
- Scan type คืออะไร?
- `Rows Removed by Filter` บอกอะไร?  
- `Execution Time` เท่าไหร่?

ให้จำตัวเลข Execution Time ตรงนี้ไว้ — เดี๋ยว Lab B จะเอามาเทียบ

### แบบฝึกหัดที่ 1: ลองเปลี่ยน driver เป็นอีกคน

In [15]:
# TODO: ลองรัน EXPLAIN ANALYZE หา driver_id = 400 ดูบ้าง
# แล้วตอบคำถาม: scan type เหมือนหรือต่างจาก driver_id = 250? execution time ใกล้เคียงกันไหม? ทำไม?

# เขียนโค้ดตรงนี้:
explain("SELECT * FROM trips WHERE driver_id = 400")


Gather  (cost=1000.00..9386.57 rows=994 width=53) (actual time=8.210..236.768 rows=1037.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared read=5683
  ->  Parallel Seq Scan on trips  (cost=0.00..8287.17 rows=414 width=53) (actual time=4.970..226.500 rows=345.67 loops=3)
        Filter: (driver_id = 400)
        Rows Removed by Filter: 166321
        Buffers: shared read=5683
Planning:
  Buffers: shared hit=71 read=10 dirtied=2
Planning Time: 5.601 ms
Execution Time: 237.538 ms



---
## Lab B: สร้าง Index และเทียบผล

สร้าง index บนคอลัมน์ `driver_id` แล้วรัน query เดิมอีกครั้ง

In [16]:
run("CREATE INDEX IF NOT EXISTS idx_trips_driver_id ON trips (driver_id)")
print("สร้าง index สำเร็จ")

Connection หลุด (SSL connection has been closed unexpectedly
) — กำลังเชื่อมต่อใหม่...
สร้าง index สำเร็จ


Neon แยก Compute ออกจาก Storage — index จะถูกเก็บอยู่ใน Storage layer เดียวกับตัวข้อมูลในตาราง trips เอง ผูกอยู่กับ branch ที่กำลังใช้งานอยู่ (เช่น branch production ที่เป็น default) และอยู่ใน database + schema เดียวกับตาราง (ปกติคือ database neondb, schema public) — ไม่ได้แยกเก็บที่อื่น

In [17]:
explain("SELECT * FROM trips WHERE driver_id = 250", label="Lab B: driver_id = 250 (มี index แล้ว)")

=== Lab B: driver_id = 250 (มี index แล้ว) ===
Bitmap Heap Scan on trips  (cost=12.13..2583.10 rows=994 width=53) (actual time=0.204..1.588 rows=998.00 loops=1)
  Recheck Cond: (driver_id = 250)
  Heap Blocks: exact=928
  Buffers: shared hit=928 read=4
  ->  Bitmap Index Scan on idx_trips_driver_id  (cost=0.00..11.88 rows=994 width=0) (actual time=0.109..0.109 rows=998.00 loops=1)
        Index Cond: (driver_id = 250)
        Index Searches: 1
        Buffers: shared read=4
Planning:
  Buffers: shared hit=54 read=14 dirtied=2
Planning Time: 7.072 ms
Execution Time: 2.217 ms



**สังเกต:** Scan type เปลี่ยนเป็น **Bitmap Heap Scan + Bitmap Index Scan** (ไม่ใช่ Index Scan ตรง ๆ)


Bitmap Scan เกิดขึ้นเมื่อ match หลายแถวที่กระจายกันในตาราง — driver คนหนึ่งมีประมาณ 1,000 เที่ยว กระจายอยู่ทั่วตาราง 500,000 แถว จึงเข้าเงื่อนไขที่ planner เลือกใช้ Bitmap Scan แทน Index Scan ตรง ๆ


**ทำไม planner ถึงเลือก Bitmap แทน Index Scan ตรง ๆ**

**ลองเทียบสองทางเลือกที่ planner ต้องเปรียบเทียบ**

* ถ้าใช้ Index Scan ตรง ๆ — เจอตำแหน่งจาก index ทีละตำแหน่ง แล้ววิ่งไปอ่าน heap ทันทีทีละแถว เจอ 1 ตำแหน่งวิ่งไปอ่าน 1 ที เจอถัดไปวิ่งไปอ่านอีกที ทำแบบนี้ซ้ำ 1,000 รอบ เพราะตำแหน่งกระจัดกระจายกันมาก การ "กระโดด" ไปมาในดิสก์แบบนี้ (random I/O) ช้า

* ถ้าใช้ Bitmap Scan — ไล่หาตำแหน่งทั้ง 1,000 ตำแหน่งจาก index ให้ครบก่อน (ยังไม่แตะ heap) เก็บเป็นแผนที่บิตไว้ แล้วเรียงลำดับตำแหน่งใหม่จากน้อยไปมาก จากนั้นค่อยเดินอ่าน heap ทีเดียวตามลำดับ ไม่กระโดดไปมา — เร็วกว่ามากเมื่อ match เยอะขนาดนี้

**การตัดสินใจของ planner**

* Planner คำนวณจาก จำนวนแถวที่ match โดยประมาณ (จาก statistics ที่เก็บไว้ตอน ANALYZE) — ถ้า match น้อยมาก (1-2 แถว) overhead ของการสร้าง bitmap ไม่คุ้ม เลือก Index Scan ตรง ๆ ดีกว่า แต่ถ้า match เยอะขนาด 1,000 แถวแบบนี้ การสร้าง bitmap แล้วเรียงลำดับก่อนอ่านคุ้มกว่าเยอะ จึงเลือก Bitmap Scan แทน

### แบบฝึกหัดที่ 2: ให้เขียนคำสั่งสำหรับดู Index Scan แบบตรง ๆ (ไม่ใช่ Bitmap) ของ trip_id

In [18]:
# TODO: ลอง EXPLAIN ANALYZE หา trip_id ที่เจาะจงตัวเดียว (เช่น trip_id = 100000)
# สังเกตว่า scan type ต่างจากตอนหา driver_id ยังไง ทำไมถึงต่างกัน?

explain("SELECT * FROM trips WHERE trip_id = 100000")

Index Scan using trips_pkey on trips  (cost=0.42..8.44 rows=1 width=53) (actual time=2.087..2.089 rows=1.00 loops=1)
  Index Cond: (trip_id = 100000)
  Index Searches: 1
  Buffers: shared hit=4 read=3
Planning:
  Buffers: shared hit=8
Planning Time: 0.091 ms
Execution Time: 2.145 ms



### แบบฝึกหัดที่ 3: ให้เขียนโค้ดคำนวณว่า Execution Time ของ Lab A (ไม่มี index) กับ Lab B (มี index) ต่างกันกี่เท่า

In [ ]:
# TODO: เขียนโค้ดคำนวณว่า Execution Time ของ Lab A (ไม่มี index) กับ Lab B (มี index)
# ต่างกันกี่เท่า โดยใช้ตัวเลขที่จดไว้จากสองเซลล์ EXPLAIN ANALYZE ก่อนหน้า


# ANSWERS

# Lab A และ Lab B ต่างกัน 1.03357 เท่า (2.217 ms /  2.145 ms)
# คิดเป็น percent = 3.36%


---
## Lab C: Sargable vs Non-sargable

การ wrap คอลัมน์ด้วยฟังก์ชันทำให้ index ใช้ไม่ได้ มาดูของจริงกัน

In [19]:
run("CREATE INDEX IF NOT EXISTS idx_trips_date ON trips (trip_date)")
print("สร้าง index บน trip_date สำเร็จ")

Connection หลุด (SSL connection has been closed unexpectedly
) — กำลังเชื่อมต่อใหม่...
สร้าง index บน trip_date สำเร็จ


::date คือ PostgreSQL casting syntax — แปลงชนิดข้อมูลจาก TIMESTAMP (ที่เก็บทั้งวันที่+เวลา เช่น 2025-06-15 14:32:07) ให้เหลือแค่ ส่วนวันที่อย่างเดียว (2025-06-15) ตัดเวลาทิ้ง

In [20]:
# Non-sargable: trip_date::date คือการ wrap คอลัมน์ด้วยฟังก์ชัน/การแปลงชนิดข้อมูล
explain(
    "SELECT * FROM trips WHERE trip_date::date = '2025-06-15'",
    label="Non-sargable: trip_date::date = '2025-06-15'"
)

=== Non-sargable: trip_date::date = '2025-06-15' ===
Gather  (cost=1000.00..10058.00 rows=2500 width=53) (actual time=0.265..53.941 rows=868.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared hit=5683
  ->  Parallel Seq Scan on trips  (cost=0.00..8808.00 rows=1042 width=53) (actual time=0.049..44.510 rows=289.33 loops=3)
        Filter: ((trip_date)::date = '2025-06-15'::date)
        Rows Removed by Filter: 166377
        Buffers: shared hit=5683
Planning:
  Buffers: shared hit=64 read=13 dirtied=2
Planning Time: 6.624 ms
Execution Time: 54.095 ms



In [21]:
# Sargable: เขียนใหม่เป็นช่วง range แทน ไม่ wrap คอลัมน์
explain(
    "SELECT * FROM trips WHERE trip_date >= '2025-06-15' AND trip_date < '2025-06-16'",
    label="Sargable: trip_date >= ... AND trip_date < ..."
)

=== Sargable: trip_date >= ... AND trip_date < ... ===
Bitmap Heap Scan on trips  (cost=22.55..2582.74 rows=988 width=53) (actual time=0.198..1.187 rows=868.00 loops=1)
  Recheck Cond: ((trip_date >= '2025-06-15 00:00:00'::timestamp without time zone) AND (trip_date < '2025-06-16 00:00:00'::timestamp without time zone))
  Heap Blocks: exact=800
  Buffers: shared hit=800 read=6
  ->  Bitmap Index Scan on idx_trips_date  (cost=0.00..22.30 rows=988 width=0) (actual time=0.126..0.126 rows=868.00 loops=1)
        Index Cond: ((trip_date >= '2025-06-15 00:00:00'::timestamp without time zone) AND (trip_date < '2025-06-16 00:00:00'::timestamp without time zone))
        Index Searches: 1
        Buffers: shared read=6
Planning:
  Buffers: shared hit=7 read=4
Planning Time: 1.898 ms
Execution Time: 1.256 ms



**ผลที่ได้:** Non-sargable ได้ **Seq Scan** (~.......ms)

ส่วน Sargable ได้ **Bitmap Heap Scan** (~......ms)

พิจารณาค่าเวลาที่แตกต่างกัน

### แบบฝึกหัดที่ 4: แก้ query ที่ไม่ sargable ให้เป็น sargable

In [ ]:
# โจทย์: query นี้ไม่ sargable เพราะ wrap คอลัมน์ fare_amount ด้วยการคำนวณ (สมมติคำนวณภาษี 7%)
# ลองรัน EXPLAIN ANALYZE เพื่อดูว่าเป็น Seq Scan จริงไหม
explain(
    "SELECT * FROM trips WHERE fare_amount * 1.07 > 200",
    label="Non-sargable: fare_amount * 1.07 > 200"
)

=== Non-sargable: fare_amount * 1.07 > 200 ===
Seq Scan on trips  (cost=0.00..13183.00 rows=166667 width=53) (actual time=0.011..94.439 rows=331954.00 loops=1)
  Filter: ((fare_amount * 1.07) > '200'::numeric)
  Rows Removed by Filter: 168046
  Buffers: shared hit=5683
Planning Time: 0.065 ms
Execution Time: 110.134 ms



In [23]:
# TODO: เขียน query ใหม่ให้ sargable (ไม่ wrap คอลัมน์ fare_amount)
# แล้วรัน explain() เทียบดู

# สร้าง index ชื่อ idx_trips_fare
run("CREATE INDEX IF NOT EXISTS idx_trips_fare ON trips (fare_amount)")

explain(
    "SELECT * FROM trips WHERE fare_amount > 200",
    label="sargable: fare_amount > 200"
)


=== sargable: fare_amount > 200 ===
Seq Scan on trips  (cost=0.00..11933.00 rows=321479 width=53) (actual time=0.013..67.071 rows=318019.00 loops=1)
  Filter: (fare_amount > '200'::numeric)
  Rows Removed by Filter: 181981
  Buffers: shared hit=5683
Planning:
  Buffers: shared hit=84 read=15 dirtied=2
Planning Time: 8.724 ms
Execution Time: 81.732 ms



---
## Composite Index & Leftmost-Prefix Rule

ลองสร้าง composite index บน `(driver_id, trip_date)` แล้วทดสอบ leftmost-prefix rule

In [ ]:
# ลบ index เดี่ยวของ driver_id และ trip_date ทิ้งก่อน เพื่อบังคับให้ planner ต้องเลือกใช้ composite index ตัวใหม่
run("DROP INDEX IF EXISTS idx_trips_driver_id")
run("DROP INDEX IF EXISTS idx_trips_date")

run("CREATE INDEX IF NOT EXISTS idx_trips_driver_date ON trips (driver_id, trip_date)")
print("สร้าง composite index สำเร็จ")

สร้าง composite index สำเร็จ


In [ ]:
# ใช้ทั้งสองคอลัมน์ (leftmost = driver_id ตรงตาม rule)
explain(
    "SELECT * FROM trips WHERE driver_id = 250 AND trip_date >= '2025-06-01' AND trip_date < '2025-07-01'",
    label="ใช้ driver_id (leftmost) + trip_date"
)

=== ใช้ driver_id (leftmost) + trip_date ===
Bitmap Heap Scan on trips  (cost=5.06..191.87 rows=50 width=53) (actual time=0.029..0.113 rows=69.00 loops=1)
  Recheck Cond: ((driver_id = 250) AND (trip_date >= '2025-06-01 00:00:00'::timestamp without time zone) AND (trip_date < '2025-07-01 00:00:00'::timestamp without time zone))
  Heap Blocks: exact=69
  Buffers: shared hit=72
  ->  Bitmap Index Scan on idx_trips_driver_date  (cost=0.00..5.05 rows=50 width=0) (actual time=0.014..0.014 rows=69.00 loops=1)
        Index Cond: ((driver_id = 250) AND (trip_date >= '2025-06-01 00:00:00'::timestamp without time zone) AND (trip_date < '2025-07-01 00:00:00'::timestamp without time zone))
        Index Searches: 1
        Buffers: shared hit=3
Planning:
  Buffers: shared hit=5
Planning Time: 0.122 ms
Execution Time: 0.141 ms



In [ ]:
# ข้ามคอลัมน์ซ้ายสุดไปเลย ใช้แค่ trip_date
explain(
    "SELECT * FROM trips WHERE trip_date >= '2025-06-01' AND trip_date < '2025-07-01'",
    label="ใช้แค่ trip_date (ข้าม leftmost)"
)

=== ใช้แค่ trip_date (ข้าม leftmost) ===
Bitmap Heap Scan on trips  (cost=2244.47..8305.55 rows=25205 width=53) (actual time=317.235..324.385 rows=25155.00 loops=1)
  Recheck Cond: ((trip_date >= '2025-06-01 00:00:00'::timestamp without time zone) AND (trip_date < '2025-07-01 00:00:00'::timestamp without time zone))
  Heap Blocks: exact=5623
  Buffers: shared hit=6622 read=701
  ->  Bitmap Index Scan on idx_trips_driver_date  (cost=0.00..2238.17 rows=25205 width=0) (actual time=316.220..316.220 rows=25155.00 loops=1)
        Index Cond: ((trip_date >= '2025-06-01 00:00:00'::timestamp without time zone) AND (trip_date < '2025-07-01 00:00:00'::timestamp without time zone))
        Index Searches: 501
        Buffers: shared hit=999 read=701
Planning Time: 0.076 ms
Execution Time: 325.538 ms



**ผลที่ควรเห็น:** แบบแรก (ใช้ `driver_id` เป็น leftmost) ได้ **Bitmap Heap Scan** เร็วมาก (<1ms) ส่วนแบบที่สอง (ข้าม `driver_id` ไปเลย) ได้ **Seq Scan** ช้ากว่ามาก  — ตรงตาม Leftmost-Prefix Rule ที่เรียนไป

### แบบฝึกหัดที่ 5: ทดสอบ Leftmost-Prefix เพิ่มเติม ให้ใช้แค่ driver_id

In [ ]:
# TODO: ลอง query ที่ใช้แค่ trip_date เพียงอย่างเดียวแบบ equality (ไม่ใช่ range)
# WHERE trip_date >= '2025-06-15' AND trip_date < '2025-06-16'  (ไม่มี driver_id)
# เทียบกับ query ที่ใช้ driver_id อย่างเดียว (ไม่มี trip_date)
# ตัวไหนใช้ index ได้ดีกว่ากัน ทำไม?
explain(
    ".................",
    label="ใช้แค่ driver_id (leftmost อย่างเดียว)"
)


---
## Lab D (เพิ่มเติม): พิจารณา Slow Query — Low Selectivity

โจทย์: query หา trip ที่ `trip_status = 'completed'` ช้า ทั้งที่สร้าง index ไว้แล้ว

In [ ]:
run("CREATE INDEX IF NOT EXISTS idx_trips_status ON trips (trip_status)")
print("สร้าง index สำเร็จ")

สร้าง index สำเร็จ


In [ ]:
explain(
    "SELECT * FROM trips WHERE trip_status = 'completed'",
    label="trip_status = 'completed' (มี index แล้ว)"
)

=== trip_status = 'completed' (มี index แล้ว) ===
Seq Scan on trips  (cost=0.00..11933.00 rows=332583 width=53) (actual time=0.013..53.042 rows=333682.00 loops=1)
  Filter: (trip_status = 'completed'::text)
  Rows Removed by Filter: 166318
  Buffers: shared hit=5683
Planning:
  Buffers: shared hit=22 read=2
Planning Time: 1.230 ms
Execution Time: 68.509 ms



**สังเกต:** แม้จะมี index แล้ว แต่ planner ก็ยังเลือก **Seq Scan** อยู่ดี! ทำไม?

In [ ]:
# ลองดูการกระจายตัวของค่าในคอลัมน์นี้
run("""
SELECT trip_status, count(*), round(100.0*count(*)/(SELECT count(*) FROM trips), 1) AS pct
FROM trips GROUP BY trip_status ORDER BY 2 DESC
""")

,trip_status,count,pct
0,completed,333682,66.7
1,in_progress,83435,16.7
2,cancelled,82883,16.6


### แบบฝึกหัดที่ 6: ให้อธิบายและเสนอทางแก้ไข

In [ ]:
# TODO: จากตัวเลขสัดส่วนที่เห็นด้านบน อธิบายว่าทำไม planner ถึงไม่เลือกใช้ index
# แล้วเสนอว่าควรทำยังไงกับ index นี้ (เก็บไว้ / ลบทิ้ง / อื่น ๆ)

run("....................")

---
---
## แบบฝึกหัดท้ายบท

โจทย์: **ระบบจองโรงแรม (Hotel Booking)**

### Schema

```sql
CREATE TABLE bookings (
    booking_id      BIGSERIAL PRIMARY KEY,
    hotel_id        INT NOT NULL,
    guest_id        INT NOT NULL,
    room_type       TEXT NOT NULL,       -- 'Standard', 'Deluxe', 'Suite', 'Villa'
    booking_status  TEXT NOT NULL,       -- 'confirmed', 'cancelled', 'pending'
    check_in_date   TIMESTAMP NOT NULL,
    total_price     NUMERIC(10,2) NOT NULL
);
```

รันเซลล์ด้านล่างเพื่อสร้างและ seed ข้อมูล 400,000 แถว (ใช้เวลาประมาณ 1-2 นาที)

In [ ]:
import random
import io
from datetime import datetime, timedelta

run("""
DROP TABLE IF EXISTS bookings;
CREATE TABLE bookings (
    booking_id      BIGSERIAL PRIMARY KEY,
    hotel_id        INT NOT NULL,
    guest_id        INT NOT NULL,
    room_type       TEXT NOT NULL,
    booking_status  TEXT NOT NULL,
    check_in_date   TIMESTAMP NOT NULL,
    total_price     NUMERIC(10,2) NOT NULL
);
""")

random.seed(7)
ROOM_TYPES = ["Standard", "Deluxe", "Suite", "Villa"]
BOOKING_STATUSES = ["confirmed", "confirmed", "confirmed", "confirmed", "cancelled", "pending"]
N_HOTELS = 200
N_GUESTS = 8000
N_BOOKINGS = 400_000

def gen_bookings_chunk(n, start=datetime(2025, 1, 1)):
    buf = io.StringIO()
    for _ in range(n):
        hotel_id = random.randint(1, N_HOTELS)
        guest_id = random.randint(1, N_GUESTS)
        room = random.choice(ROOM_TYPES)
        status = random.choice(BOOKING_STATUSES)
        d = start + timedelta(days=random.randint(0, 500), seconds=random.randint(0, 86399))
        price = round(random.uniform(800, 15000), 2)
        buf.write(f"{hotel_id}\t{guest_id}\t{room}\t{status}\t{d}\t{price}\n")
    buf.seek(0)
    return buf

t0 = time.time()
total = 0
CHUNK = 50_000
with conn.cursor() as cur:
    while total < N_BOOKINGS:
        n = min(CHUNK, N_BOOKINGS - total)
        buf = gen_bookings_chunk(n)
        cur.copy_expert(
            "COPY bookings (hotel_id, guest_id, room_type, booking_status, check_in_date, total_price) FROM STDIN",
            buf
        )
        total += n

run("ANALYZE bookings")
print(f"Seed bookings เสร็จใน {time.time()-t0:.1f} วินาที ({total:,} แถว)")

Seed bookings เสร็จใน 9.9 วินาที (400,000 แถว)


### โจทย์ที่ 1: Baseline & Index พื้นฐาน

หา booking ทั้งหมดของ `hotel_id = 50` — ให้วัดผลทั้งตอนยังไม่มี index และเมื่อมีการสร้าง index แล้วให้วัดผลอีกครั้ง เพื่อเปรียบเทียบกัน

In [ ]:
# TODO: EXPLAIN ANALYZE หา hotel_id = 50 (ยังไม่มี index)
run("DROP INDEX IF EXISTS idx_bookings_hotel")

explain("..........................", label="ก่อนมี index")

run("CREATE INDEX .............................")

explain("..........................", label="หลังมี index")

### โจทย์ที่ 2: หา Non-sargable Query แล้วแก้ไข

Query นี้ต้องการหา booking ที่ check-in ใน**เดือนมิถุนายน** เท่านั้น โดยทีมเดิมเขียนไว้แบบนี้

```sql
SELECT * FROM bookings WHERE EXTRACT(MONTH FROM check_in_date) = 6
```

1. รัน `EXPLAIN ANALYZE` แล้วสังเกตว่า sargable หรือไม่
2. อธิบายว่าทำไม
3. ถ้าโจทย์จริง ๆ ต้องการแค่เดือนมิถุนายนของปี 2025 เท่านั้น (ไม่ใช่ทุกปี) ให้เขียน query ใหม่ให้ sargable

In [ ]:
# TODO: ทำตามขั้นตอนที่ 1-3 ด้านบน
run("CREATE INDEX idx_bookings_checkin ON bookings (check_in_date)")
explain(
    "SELECT * FROM bookings WHERE EXTRACT(MONTH FROM check_in_date) = 6",
    label="Non-sargable: EXTRACT(MONTH FROM check_in_date)"
)


=== Non-sargable: EXTRACT(MONTH FROM check_in_date) ===
Gather  (cost=1000.00..7776.00 rows=2000 width=47) (actual time=0.291..84.528 rows=24104.00 loops=1)
  Workers Planned: 2
  Workers Launched: 2
  Buffers: shared hit=4076
  ->  Parallel Seq Scan on bookings  (cost=0.00..6576.00 rows=833 width=47) (actual time=0.034..48.404 rows=8034.67 loops=3)
        Filter: (EXTRACT(month FROM check_in_date) = '6'::numeric)
        Rows Removed by Filter: 125299
        Buffers: shared hit=4076
Planning:
  Buffers: shared hit=17 read=5
Planning Time: 2.006 ms
Execution Time: 89.664 ms



In [ ]:
# แก้เป็น sargable โดยระบุช่วงวันที่ตรง ๆ
explain(
    "..................................",
    label="Sargable: ระบุช่วงวันที่ตรง ๆ"
)

### โจทย์ที่ 3: พิจารณา Selectivity

เช็คการกระจายตัวของ `booking_status` แล้ววิเคราะห์ว่าควรสร้าง index บนคอลัมน์นี้ไหม พร้อมให้เหตุผล

In [ ]:
# TODO: query หาสัดส่วนของแต่ละค่าใน booking_status
run("""
SELECT booking_status, count(*),
       round(100.0*count(*)/(SELECT count(*) FROM bookings), 1) AS pct
FROM bookings GROUP BY booking_status ORDER BY 2 DESC
""")

,booking_status,count,pct
0,confirmed,267002,66.8
1,cancelled,66718,16.7
2,pending,66280,16.6


ควรสร้าง index บน booking_status ทั้ง column หรือไม่ ???

### โจทย์ที่ 4: ออกแบบ Composite Index

Business ต้องการ query แบบนี้บ่อยที่สุด คือ **"หา booking ของโรงแรมหนึ่ง ๆ ในช่วงวันที่ที่กำหนด"** (`WHERE hotel_id = ? AND check_in_date BETWEEN ? AND ?`)

1. ออกแบบ composite index ที่เหมาะกับ query pattern นี้ (ต้องตัดสินใจว่าคอลัมน์ไหนควรอยู่ซ้ายสุด)
2. ทดสอบด้วย `EXPLAIN ANALYZE`
3. ทดสอบกรณี query แค่ `check_in_date` อย่างเดียว (ข้าม `hotel_id`) แล้วอธิบายผลที่เกิดขึ้น

In [ ]:
# TODO: ทำตามขั้นตอนที่ 1-3
# ขั้นตอน 1: ออกแบบ index — hotel_id เป็น equality (leftmost) แล้วตามด้วย check_in_date เป็น range
run("DROP INDEX IF EXISTS idx_bookings_hotel")
run("DROP INDEX IF EXISTS idx_bookings_checkin")
run("CREATE INDEX idx_bookings_hotel_checkin ................................")

# ขั้นตอน 2: ทดสอบ query ตาม pattern ที่ business ต้องการ
explain(
    ".......................................",
    label="hotel_id (leftmost) + check_in_date"
)

In [ ]:
# ขั้นตอน 3: ทดสอบข้าม leftmost
explain(
    ".......................................",
    label="check_in_date อย่างเดียว (ข้าม leftmost)"
)

### โจทย์ที่ 5: สรุป

จากทั้ง 4 โจทย์ข้างต้น ให้เขียนสรุปทั้ง 3 ข้อนี้
1. ควรสร้าง index อะไรบ้างบนตาราง `bookings`
2. มี index ไหนที่ไม่ควรสร้าง (หรือควรลบถ้าเคยสร้างไปแล้ว)

เขียนคำตอบ...

---
## จบ Lab Chapter 6

**สิ่งที่ได้ฝึกไปทั้งหมด**
- วัดผล query ก่อน-หลังสร้าง Index ด้วย `EXPLAIN ANALYZE`
- แยกแยะและแก้ไข Non-sargable query
- ออกแบบ Composite Index ตาม query pattern จริง
- บอกได้ว่าเมื่อไหร่ไม่ควรสร้าง Index (low selectivity)
- ประยุกต์ทักษะทั้งหมดกับ domain ใหม่ที่ไม่เคยเห็นมาก่อน (Hotel Booking)
